# Feature Engineering

With the objective of preparing the previously analyzed data from the **TCGA Glioblastoma pipeline-processed dataset** for **unsupervised learning** models such as, **K-Means and Hierarchical Clustering** as well as data compression strategies such as **Principal Component Analysis** due to the high-dimensional nature of the dataset, the present dataset must now undergo a feature engineering process which will ensure that the data can be efficiently processed by the previously stated unsupervised learning techniques. 

As we saw in the previous EDA phase, most of the features have variance and mean of 0, which means they are practically "turned off" in the context of Glioblastoma or brain tissue, therefore, in order to select the most relevant features for clustering segmentation, we must prioritize maintaining the most variable features in the following stages of the investigation. In accordance to the nature of unsupervised learning, the dataset does not contain any response variable which may help us classify patients or generate regression models of the data, therefore, the present study will prioritize general variance in the features of the dataset to then generate relevant clusters in the context of the data. 

## Normalization and Scaling

The TCGA GBM data used in this project has, as previously stated, been processed using the STAR-TPM pipeline, which is a method of *biological normalization*. This means that the feature gene expression data has been adjusted for gene length and sequencing length, which enables the comparison of sample-wide data. Distance-based clustering algorithms, such as **K-means or Hierarchical Clustering** will not be able to detect nuances in different genetic expressions for specific features; any given important features for clustering may variate highly in different scales, and therefore, any of the previously stated models may prioritize a specific feature due to its higher magnitude scale variance. Therefore, the data must still be normalized, so as to give every feature in the dataset the same importance before the stated unsupervised learning methods.

The decisions to be taken in accordance to the previous considerations are the following:

- **Log-normalization**: As we saw in the EDA stage, specifically in the skewness histogram for the data, features in this dataset tend to have right-skewness, which means that for specific features in the dataset, irregularities in genetic expression data may cause statistically significant spikes in some patient samples, which might reduce the importance of variability in the same feature for other samples. The solution to this is to apply log-normalization, a skill which entails pulling the stated extreme values inwards, making the distribution for each of the gene features much closer to a normal bell curve, which will be of benefit for the learning models.
- **Standard scaling**: Even after log-normalization, the problem of different scales for distinct features with similar variance still applies. Learning models will automatically prioritize higher scales of variance due to their numerical importance. To mitigate this problem, standard scaling, which entails scaling every feature to have a mean of 0 and a standard deviation of 1 will ensure that variability is the driving factor in decision making for the unsupervised learning models.

The previously stated steps will mitigate the previously stated problems and ensure that the data is ready to be analyzed using unsupervised learning. The following code accomplishes the normalization and scaling steps.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

X = pd.read_csv('../../data/star_data/X.csv')

# --- STEP 1: Log Transformation ---
# Apply log2(x + 1) to the entire dataset to compress extreme tails
# We use +1 to ensure that 0 TPM remains 0 (since log2(1) = 0)
df_log = np.log2(X + 1)
print("1. Log transformation applied.")

# --- STEP 2: Variance Filtering (Feature Reduction) ---
# Calculate variance on the LOG-TRANSFORMED data
gene_variances = df_log.var()

# Isolate the top 5,000 most variable genes
top_5000_hvg = gene_variances.sort_values(ascending=False).head(5000).index

# Subset the dataframe to only include these genes
df_filtered = df_log[top_5000_hvg]
print(f"2. Features reduced. New matrix shape: {df_filtered.shape}")

# --- STEP 3: Standard Scaling (Z-Score) ---
# Initialize the scaler
scaler = StandardScaler()

# Fit and transform the subsetted, log-transformed data
scaled_matrix = scaler.fit_transform(df_filtered)

# Rebuild the final DataFrame, keeping patient barcodes (index) and gene names (columns) safe
df_final = pd.DataFrame(
    scaled_matrix, 
    index=df_filtered.index, 
    columns=df_filtered.columns
)
print("3. Standard scaling applied. Data is ready for PCA.")

FileNotFoundError: [Errno 2] No such file or directory: '../../star_data/X.csv'